# Experiment 9A: Deduplication Audit

Verify zero data leakage between aggregated training data and FPB test set.
Three checks: exact match, fuzzy match (>90%), semantic near-duplicates (cosine >0.95).

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from difflib import SequenceMatcher
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import json
import time

## 1. Load Data

In [ ]:
FPB_SOURCE = 5

ds = load_dataset("neoyipeng/financial_reasoning_aggregated")
ds_sentiment = ds.filter(lambda x: x["task"] == "sentiment")
ds_no_fpb = ds_sentiment.filter(lambda x: x["source"] != FPB_SOURCE)

train_texts = [row['text'] for row in ds_no_fpb['train']]
val_texts = [row['text'] for row in ds_no_fpb['validation']]
all_train = train_texts + val_texts

def load_fpb_file(agree_level):
    """Load FPB from zip file (works with datasets>=4.0)."""
    from huggingface_hub import hf_hub_download
    import zipfile, os
    path = hf_hub_download("financial_phrasebank", "data/FinancialPhraseBank-v1.0.zip", repo_type="dataset")
    extract_dir = "/tmp/fpb_data"
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(path, "r") as z:
        z.extractall(extract_dir)
    filename = {"50agree": "Sentences_50Agree.txt", "allagree": "Sentences_AllAgree.txt"}[agree_level]
    filepath = os.path.join(extract_dir, "FinancialPhraseBank-v1.0", filename)
    sentences, labels = [], []
    label_map = {"negative": 0, "neutral": 1, "positive": 2}
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            at_idx = line.rfind("@")
            if at_idx == -1:
                continue
            label_str = line[at_idx+1:].strip().lower()
            if label_str in label_map:
                sentences.append(line[:at_idx].strip())
                labels.append(label_map[label_str])
    return sentences, labels

def load_fpb(agree_level="50agree"):
    """Load FPB with fallback for datasets>=4.0."""
    try:
        config = f"sentences_{agree_level}"
        ds = load_dataset("financial_phrasebank", config, trust_remote_code=True)["train"]
        return ds["sentence"], ds["label"]
    except Exception:
        return load_fpb_file(agree_level)

fpb_texts, fpb_labels = load_fpb("50agree")

print(f"Training samples (excl FPB): {len(all_train)}")
print(f"FPB 50agree samples: {len(fpb_texts)}")

## 2. Exact Match Check

In [ ]:
def normalize(text):
    return text.strip().lower()

train_norm = set(normalize(t) for t in all_train)
fpb_norm = [(i, normalize(t)) for i, t in enumerate(fpb_texts)]

exact_matches = [(i, t) for i, t in fpb_norm if t in train_norm]
print(f"Exact matches: {len(exact_matches)} / {len(fpb_texts)}")
for i, t in exact_matches[:10]:
    print(f"  [{i}] {t[:100]}...")

## 3. Fuzzy Match Check (>90% similarity)

In [ ]:
def fuzzy_ratio(a, b):
    return SequenceMatcher(None, a, b).ratio()

start = time.time()
fuzzy_matches = []
train_norm_list = list(train_norm)

for idx, (i, fpb_t) in enumerate(fpb_norm):
    if idx % 500 == 0:
        print(f"  Checking FPB sample {idx}/{len(fpb_norm)}...")
    fpb_len = len(fpb_t)
    for train_t in train_norm_list:
        if abs(len(train_t) - fpb_len) > 20:
            continue
        ratio = fuzzy_ratio(fpb_t, train_t)
        if ratio > 0.90:
            fuzzy_matches.append((i, fpb_t[:80], train_t[:80], ratio))
            break

elapsed = time.time() - start
print(f"\nFuzzy matches (>90%): {len(fuzzy_matches)} ({elapsed:.1f}s)")
for idx, fpb_s, train_s, r in fuzzy_matches[:10]:
    print(f"  [{idx}] ratio={r:.3f}")
    print(f"    FPB:   {fpb_s}")
    print(f"    Train: {train_s}")

## 4. Semantic Near-Duplicate Check (cosine > 0.95)

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding FPB texts...")
fpb_embs = model.encode(fpb_texts, batch_size=256, show_progress_bar=True)
print("Encoding training texts...")
train_embs = model.encode(all_train, batch_size=256, show_progress_bar=True)

THRESHOLD = 0.95
semantic_matches = []
chunk_size = 500
for start in range(0, len(fpb_embs), chunk_size):
    end = min(start + chunk_size, len(fpb_embs))
    sims = cosine_similarity(fpb_embs[start:end], train_embs)
    for i in range(end - start):
        max_sim = sims[i].max()
        if max_sim > THRESHOLD:
            best_j = sims[i].argmax()
            semantic_matches.append((
                start + i, int(best_j), float(max_sim),
                fpb_texts[start + i][:80],
                all_train[best_j][:80]
            ))

print(f"\nSemantic near-duplicates (cosine > {THRESHOLD}): {len(semantic_matches)}")
for fpb_i, train_j, sim, fpb_s, train_s in semantic_matches[:10]:
    print(f"  sim={sim:.4f}")
    print(f"    FPB:   {fpb_s}")
    print(f"    Train: {train_s}")

## 5. Summary

In [ ]:
print("=" * 60)
print("DEDUPLICATION AUDIT SUMMARY")
print("=" * 60)
print(f"Training set size:           {len(all_train)}")
print(f"FPB test set size:           {len(fpb_texts)}")
print(f"Exact matches:               {len(exact_matches)}")
print(f"Fuzzy matches (>90%):        {len(fuzzy_matches)}")
print(f"Semantic near-dupes (>0.95): {len(semantic_matches)}")
print()

total_contaminated = len(exact_matches) + len(fuzzy_matches) + len(semantic_matches)
if total_contaminated == 0:
    print("CLEAN: No data leakage detected.")
else:
    print(f"WARNING: {total_contaminated} potential leakage samples found.")
    print("Remove these before continuing with Experiments B-E.")

results = {
    "training_size": len(all_train),
    "fpb_size": len(fpb_texts),
    "exact_matches": len(exact_matches),
    "fuzzy_matches": len(fuzzy_matches),
    "semantic_matches": len(semantic_matches),
    "total_contaminated": total_contaminated,
    "is_clean": total_contaminated == 0,
    "exact_details": [(i, t[:100]) for i, t in exact_matches],
    "fuzzy_details": [(idx, fpb_s, train_s, r) for idx, fpb_s, train_s, r in fuzzy_matches],
    "semantic_details": [(fi, tj, sim, fs, ts) for fi, tj, sim, fs, ts in semantic_matches],
}

with open("dedup_audit_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nResults saved to dedup_audit_results.json")